# Day 16 — Feature Selection 🎯

## 📌 Introduction

Feature Selection is the process of selecting the most relevant features from a dataset while removing unnecessary or less informative features.

In this project, the Titanic dataset is used with `SelectKBest` and the ANOVA F-test to identify important features for predicting passenger survival.

## 🎯 Objectives

- Understand Feature Selection
- Identify important features
- Use `SelectKBest`
- Understand the ANOVA F-test
- Compare all features with selected features
- Evaluate model performance

## 💡 Why Feature Selection?

Using unnecessary features can make a model more complex and may introduce noise.

Feature Selection can help:

- Reduce the number of features
- Simplify the model
- Improve computational efficiency
- Remove less informative variables
- Improve model interpretability

## 📊 Dataset

This project uses the Titanic dataset.

### Target Variable

`survived`

- `0` → Not Survived
- `1` → Survived

Additional features are created during the project, including `FamilySize`, `IsAlone`, `FarePerPerson`, `Title`, and `AgeGroup`.

## 🛠️ Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

## 🔍 Load Dataset

In [ ]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
)

print("Dataset Shape:", df.shape)
display(df.head())

## 🔎 Explore the Dataset

In [ ]:
print("Dataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nTarget Distribution:")
print(df["survived"].value_counts())

## 🔧 Feature Engineering

Before feature selection, some useful features are created from the original Titanic columns.

In [ ]:
df["FamilySize"] = df["sibsp"] + df["parch"] + 1

df["IsAlone"] = (
    df["FamilySize"] == 1
).astype(int)

df["FarePerPerson"] = (
    df["fare"] / df["FamilySize"]
)

df["Title"] = (
    df["who"].astype(str).str.title()
)

df["AgeGroup"] = pd.cut(
    df["age"],
    bins=[0, 12, 18, 35, 60, 100],
    labels=["Child", "Teen", "Young Adult", "Adult", "Senior"]
)

display(
    df[
        [
            "FamilySize",
            "IsAlone",
            "FarePerPerson",
            "Title",
            "AgeGroup"
        ]
    ].head(10)
)

## 📈 Explore an Engineered Feature

In [ ]:
plt.figure(figsize=(7, 5))

df.groupby(
    "FamilySize",
    observed=True
)["survived"].mean().head(8).plot(kind="bar")

plt.xlabel("Family Size")
plt.ylabel("Survival Rate")
plt.title("Survival Rate by Family Size")
plt.tight_layout()
plt.show()

## 🎯 Select Features and Target

The target is `survived`. The selected input features include both original and engineered features.

In [ ]:
features = [
    "pclass",
    "sex",
    "age",
    "fare",
    "embarked",
    "FamilySize",
    "IsAlone",
    "FarePerPerson",
    "Title",
    "AgeGroup"
]

X = df[features]
y = df["survived"]

numeric_features = [
    "pclass",
    "age",
    "fare",
    "FamilySize",
    "IsAlone",
    "FarePerPerson"
]

categorical_features = [
    "sex",
    "embarked",
    "Title",
    "AgeGroup"
]

print("Selected Input Features:")
print(features)

## ✂️ Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Samples:", len(X_train))
print("Testing Samples :", len(X_test))

## 🧹 Preprocessing

Numerical features are imputed and scaled. Categorical features are imputed and one-hot encoded.

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

## 🤖 Baseline Model — All Features

First, Logistic Regression is trained using all processed features. This gives us a baseline for comparison.

In [ ]:
baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)

print(f"Baseline Accuracy: {baseline_accuracy:.4f}")

## 🔎 Feature Selection Method

### SelectKBest

`SelectKBest` selects the top K features according to a scoring function.

### ANOVA F-test

`f_classif` calculates ANOVA F-scores for classification features. A higher score indicates a stronger statistical relationship with the target for this selection procedure.

In this project, the top **10 processed features** are selected.

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print(
    "Total Features After Encoding:",
    len(feature_names)
)

k = 10

selector = SelectKBest(
    score_func=f_classif,
    k=k
)

X_train_selected = selector.fit_transform(
    X_train_processed,
    y_train
)

X_test_selected = selector.transform(
    X_test_processed
)

selected_features = feature_names[
    selector.get_support()
]

print(f"\nTop {k} Selected Features:")

for feature in selected_features:
    print("-", feature)

## 📊 Feature Scores

In [ ]:
feature_scores = pd.DataFrame({
    "Feature": feature_names,
    "F_Score": selector.scores_
}).sort_values(
    by="F_Score",
    ascending=False
)

display(
    feature_scores.head(10)
)

## 📊 Visualize Top 10 Features

In [ ]:
top_features = (
    feature_scores
    .head(10)
    .sort_values("F_Score")
)

plt.figure(figsize=(9, 6))

plt.barh(
    top_features["Feature"],
    top_features["F_Score"]
)

plt.xlabel("F-Score")
plt.ylabel("Feature")
plt.title("Top 10 Features by ANOVA F-Score")

plt.tight_layout()
plt.show()

## 🤖 Train Model Using Selected Features

The selected features are now used to train a second Logistic Regression model.

In [ ]:
selected_model = LogisticRegression(
    max_iter=1000
)

selected_model.fit(
    X_train_selected,
    y_train
)

selected_pred = selected_model.predict(
    X_test_selected
)

selected_accuracy = accuracy_score(
    y_test,
    selected_pred
)

print(
    f"Selected Features Accuracy: "
    f"{selected_accuracy:.4f}"
)

## 🔬 Compare All Features vs Selected Features

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "All Features",
        "Selected Features"
    ],
    "Accuracy": [
        baseline_accuracy,
        selected_accuracy
    ]
})

display(comparison)

difference = (
    selected_accuracy -
    baseline_accuracy
)

print(
    f"Accuracy Difference: "
    f"{difference:.4f}"
)

## 📋 Classification Report

In [ ]:
print(
    classification_report(
        y_test,
        selected_pred
    )
)

## 🔲 Confusion Matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    selected_pred
)

print("Confusion Matrix:")
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Not Survived",
        "Survived"
    ]
)

disp.plot()

plt.title(
    "Feature Selection - Confusion Matrix"
)

plt.tight_layout()
plt.show()

## 📈 Model Comparison Visualization

In [ ]:
plt.figure(figsize=(7, 5))

plt.bar(
    comparison["Model"],
    comparison["Accuracy"]
)

plt.ylim(0, 1)

plt.ylabel("Accuracy")
plt.title("All Features vs Selected Features")

plt.tight_layout()
plt.show()

## 🔑 Key Findings

- Feature Selection reduced the number of processed input features.
- `SelectKBest` identified the top features using ANOVA F-scores.
- The selected-feature model was compared with the all-feature baseline.
- The confusion matrix helped analyze classification errors.
- Feature Selection can make a machine learning workflow more focused and interpretable.

## 🧠 Key Learnings

Through this project, I learned:

- What Feature Selection means
- Why unnecessary features can be removed
- How `SelectKBest` works
- How the ANOVA F-test can be used for classification feature selection
- How to compare models before and after feature selection
- How feature selection can affect model performance

## 🏁 Conclusion

Feature Selection is an important part of a machine learning workflow.

In this project, `SelectKBest` with the ANOVA F-test was used to select the top features from the processed Titanic dataset. A Logistic Regression model using the selected features was then compared with a model using all processed features.

## 📚 References

- Scikit-learn — SelectKBest
  https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html

- Scikit-learn — f_classif
  https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.f_classif.html

- Scikit-learn — Logistic Regression
  https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression

- Scikit-learn — StandardScaler
  https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html

- Seaborn Data Repository — Titanic Dataset
  https://github.com/mwaskom/seaborn-data

## 📅 30 Days of Machine Learning

**Day 16/30 — Feature Selection 🎯**

Learning → Experimenting → Understanding → Building 🚀